In [ ]:
#####
#  conda activate dsai-m3
#  pip install python-dotenv
#  pip install yfinance pandas numpy


In [ ]:
import sys
import logging
import os
import zipfile

from pathlib import Path
from dotenv import load_dotenv

# Ensure the log directory exists
log_dir = Path('../log')
log_dir.mkdir(parents=True, exist_ok=True)

# Configure the logging system
logging.basicConfig(
    filename=log_dir / 'app.log',
    filemode='a',  # 'a' to append, 'w' to overwrite
    format='%(asctime)s [%(levelname)s] %(message)s'
)

# Configure the logging system
logging.basicConfig(
    filename='../log/app.log',
    filemode='a', # 'a' to append, 'w' to overwrite
    format='%(asctime)s [%(levelname)s] %(message)s',
    level=logging.INFO,
    force=True  # Essential in Jupyter to override default notebook handlers
)

# load_dotenv() reads the ../.env file and populates os.environ
load_dotenv()

# GCP_PROJECT_ID     = os.getenv('GCP_PROJECT_ID')
# BUCKET_NAME        = os.getenv('BUCKET_NAME')
# LOCATION           = os.getenv('LOCATION')

# print (f"GCP Project ID: {GCP_PROJECT_ID}")
# print (f"Bucket Name: {BUCKET_NAME}")
# print (f"Location: {LOCATION}")

In [ ]:
# Data Ingestion

from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

# Ensure the 'data' directory exists
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

# Load Data
df = yf.download("SPY", start="2010-01-01", end="2026-01-01")
df = df.droplevel(1, axis=1) if isinstance(df.columns, pd.MultiIndex) else df

# Feature Engineering (Strictly Lagged)
df["Daily_Return"] = df["Close"].pct_change()
df["SMA_20"] = df["Close"].rolling(window=20).mean()
df["SMA_50"] = df["Close"].rolling(window=50).mean()
df["Volatility_20"] = df["Daily_Return"].rolling(window=20).std()

# Simple RSI calculation
delta = df["Close"].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df["RSI_14"] = 100 - (100 / (1 + rs))

# Shift all features by 1 to ensure tomorrow's prediction only uses today's close data
feature_cols = ["Daily_Return", "SMA_20", "SMA_50", "Volatility_20", "RSI_14"]
df[feature_cols] = df[feature_cols].shift(1)

# Save the processed dataframe to CSV inside the 'data' folder
output_file = data_dir / "spy_features.csv"
df.to_csv(output_file)

print(f"Data successfully saved to {output_file}")

In [ ]:
#Target Definition & Class Imbalance

# Future 5-day return (using negative shift for the future target)
df["Future_Return_5d"] = df["Close"].pct_change(5).shift(-5)

def define_target(ret):
  if ret > 0.01:
  # Buy (Class 1)
    return 1
  elif ret < -0.01:
  # Sell (Class -1 or 2)
    return -1
  else:
  # Hold (Class 0)
    return 0

df["Target"] = df["Future_Return_5d"].apply(define_target)
df.dropna(inplace=True)

# Save the final dataset with targets included
final_output = data_dir / "spy_labeled_features.csv"
df.to_csv(final_output)

# Check class proportions
print(df["Target"].value_counts(normalize=True))

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Define features and target
feature_cols = ["Daily_Return", "SMA_20", "SMA_50", "Volatility_20", "RSI_14"]
X = df[feature_cols]
y = df["Target"]

# Time-series split (crucial to prevent look-ahead bias)
train_mask = df.index < "2023-01-01"
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[~train_mask], y[~train_mask]

# Train model (class_weight="balanced" helps if classes are imbalanced)
model = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)
print(classification_report(y_test, predictions))